In [1]:
from pyspark.sql import SparkSession

# Volvemos a conectar
spark = SparkSession.builder \
    .appName("Analisis_Mercado_Laboral_Real") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

# LEER FORZANDO EL ESQUEMA NUEVO
# Usamos sampleSize para que Spark revise los documentos y vea que ahora hay 'puesto' en lugar de 'identificador'
df = spark.read.format("mongodb") \
    .option("spark.mongodb.read.inferSchema.samplingSize", "100") \
    .load()

# Ver el nuevo esquema y los datos
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- empresa: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- fecha_publicacion: string (nullable = true)
 |-- fuente: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- modalidad: string (nullable = true)
 |-- puesto: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- url_oferta: string (nullable = true)

+--------------------+-----------+--------------------+-------------------+-----------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                 _id|  categoria|             empresa|      fecha_captura|fecha_publicacion|              fuente|          grupo|           modalidad|              puesto|             salario|           ubicacion|          url_oferta|
+--------------------+-----------+---

In [2]:
# Filtrar para quedarnos solo con puestos reales y empresas válidas
df_limpio = df.filter(
    (df.puesto != "All jobs › Progra...") & 
    (df.empresa != "Programming jobs")
)

# Mostrar los datos ya limpios
df_limpio.select("puesto", "empresa", "salario", "modalidad").show(10, truncate=False)

+--------------------------------------------------------+--------------------------------------------------+---------------------+--------------------------------------------------+
|puesto                                                  |empresa                                           |salario              |modalidad                                         |
+--------------------------------------------------------+--------------------------------------------------+---------------------+--------------------------------------------------+
|Senior Front-end Developer Freelance                    |Sanctuary Computer Remote (Asia and South America)|1400 - 2000 USD/month|Sanctuary Computer Remote (Asia and South America)|
|Senior Front-end Developer Freelance                    |Sanctuary Computer Remote (Asia and South America)|                     |Sanctuary Computer Remote (Asia and South America)|
|Senior Front-end Developer Freelance                    |Sanctuary Computer Remote (

In [3]:
from pyspark.sql.functions import col, when, lower

# 1. Eliminar filas duplicadas (importante en scraping)
df_unico = df.dropDuplicates(["puesto", "empresa"])

# 2. Clasificar el nivel del cargo (Seniority)
# Usamos 'when' para crear una nueva columna basada en palabras clave
df_final = df_unico.withColumn("nivel", 
    when(lower(col("puesto")).contains("senior"), "Senior")
    .when(lower(col("puesto")).contains("semi senior") | lower(col("puesto")).contains("ssr"), "Semi-Senior")
    .when(lower(col("puesto")).contains("junior") | lower(col("puesto")).contains("jr"), "Junior")
    .otherwise("No especificado")
)

# 3. Mostrar el resultado limpio y organizado
df_final.select("puesto", "empresa", "nivel", "modalidad").show(truncate=False)

+-------------------------------------------------------+-------------------------------------------------------+---------------+-------------------------------------------------------+
|puesto                                                 |empresa                                                |nivel          |modalidad                                              |
+-------------------------------------------------------+-------------------------------------------------------+---------------+-------------------------------------------------------+
|AI Developer (Senior) Full time                        |Improving South America Remote (9 locations)           |Senior         |Improving South America Remote (9 locations)           |
|AI Engineer (Senior) – GenAI / Llms Full time          |Niuro Remote (7 locations)                             |Senior         |Niuro Remote (7 locations)                             |
|All jobs › Programming                                 |Programming j

In [4]:
from pyspark.sql.functions import col, when, lower, regexp_replace

# 1. Filtro de seguridad: Quitar filas que no son empleos reales
df_pro = df_final.filter(col("puesto") != "All jobs › Programming")

# 2. Mejorar la clasificación de Niveles (añadiendo más lógica)
df_pro = df_pro.withColumn("nivel_corregido", 
    when(lower(col("puesto")).contains("senior") | lower(col("puesto")).contains("sr"), "Senior")
    .when(lower(col("puesto")).contains("semi senior") | lower(col("puesto")).contains("ssr"), "Semi-Senior")
    .when(lower(col("puesto")).contains("junior") | lower(col("puesto")).contains("jr"), "Junior")
    .when(lower(col("puesto")).contains("lead") | lower(col("puesto")).contains("arquitecto"), "Lead/Architect")
    .otherwise("Mid/Sin especificar")
)

# 3. Limpiar la columna Modalidad (quitar el nombre de la empresa si aparece ahí)
# Esto busca palabras clave comunes
df_pro = df_pro.withColumn("modalidad_simple", 
    when(lower(col("modalidad")).contains("remote") | lower(col("modalidad")).contains("remoto"), "Remoto")
    .when(lower(col("modalidad")).contains("hybrid") | lower(col("modalidad")).contains("híbrido"), "Híbrido")
    .when(lower(col("modalidad")).contains("in-office") | lower(col("modalidad")).contains("presencial"), "Presencial")
    .otherwise("Por definir")
)

# Mostrar el resultado final pulido
df_pro.select("puesto", "empresa", "nivel_corregido", "modalidad_simple").show(20, truncate=False)

+-------------------------------------------------------+-------------------------------------------------------+-------------------+----------------+
|puesto                                                 |empresa                                                |nivel_corregido    |modalidad_simple|
+-------------------------------------------------------+-------------------------------------------------------+-------------------+----------------+
|AI Developer (Senior) Full time                        |Improving South America Remote (9 locations)           |Senior             |Remoto          |
|AI Engineer (Senior) – GenAI / Llms Full time          |Niuro Remote (7 locations)                             |Senior             |Remoto          |
|Analista Programador COBOL (Mainframe IBM) Full time   |BC Tecnología Santiago (Hybrid)                        |Mid/Sin especificar|Híbrido         |
|Analista Programador Full time                         |BC Tecnología Santiago (Hybrid)      

In [5]:
print("--- RESUMEN DEL MERCADO LABORAL ---")
# 1. ¿Cuál es el nivel de experiencia más pedido?
df_pro.groupBy("nivel_corregido").count().orderBy("count", ascending=False).show()

# 2. ¿Qué modalidad predomina en programación?
df_pro.groupBy("modalidad_simple").count().orderBy("count", ascending=False).show()

--- RESUMEN DEL MERCADO LABORAL ---
+-------------------+-----+
|    nivel_corregido|count|
+-------------------+-----+
|Mid/Sin especificar|   93|
|             Senior|   77|
|     Lead/Architect|    6|
|             Junior|    2|
+-------------------+-----+

+----------------+-----+
|modalidad_simple|count|
+----------------+-----+
|          Remoto|  114|
|         Híbrido|   57|
|     Por definir|    7|
+----------------+-----+



In [6]:
from pyspark.sql.functions import col

# Filtramos: Solo registros con puesto real y que no sean la fila 'All jobs'
df_filtrado = df_pro.filter(
    (col("puesto").isNotNull()) & 
    (col("puesto") != "All jobs › Programming")
)

print("PASO 3: Limpieza completada.")
print(f"Registros originales: {df.count()}")
print(f"Registros válidos: {df_filtrado.count()}")

PASO 3: Limpieza completada.
Registros originales: 365
Registros válidos: 178


In [7]:
#¿Qué empresa ofrece qué puesto?
df_filtrado.select("puesto", "empresa").show(10, truncate=False)

+--------------------------------------------------------------------+--------------------------------------------+
|puesto                                                              |empresa                                     |
+--------------------------------------------------------------------+--------------------------------------------+
|Developer Senior Android (SDK) & TypeScript Specialist Freelance    |FoodTag Santiago (Hybrid)                   |
|Java Developer Senior Full time                                     |Improving South America Remote (5 locations)|
|Developer Back-end Node.js Senior (Latam) Full time                 |BC Tecnología Remote                        |
|Full-Stack Engineer — Document Generation & Report Systems Full time|Blazestack Remote                           |
|Dev Integraciones Golang+Java Full time                             |Coderslab.io Remote                         |
|Ingeniero de Software Full-Stack Freelance                          |BC

In [8]:
#Solo ofertas de modalidad "Remoto".
df_filtrado.filter(col("modalidad_simple") == "Remoto").show(5)

+--------------------+-----------+--------------------+-------------------+-----------------+--------------------+---------------+--------------------+--------------------+-------+--------------------+----------+---------------+-------------------+----------------+
|                 _id|  categoria|             empresa|      fecha_captura|fecha_publicacion|              fuente|          grupo|           modalidad|              puesto|salario|           ubicacion|url_oferta|          nivel|    nivel_corregido|modalidad_simple|
+--------------------+-----------+--------------------+-------------------+-----------------+--------------------+---------------+--------------------+--------------------+-------+--------------------+----------+---------------+-------------------+----------------+
|69d65ee9222677856...|Programming|Improving South A...|2026-04-08 13:57:32|                 |https://www.geton...|mercado laboral|Improving South A...|AI Developer (Sen...|       |Improving South A...| 

In [11]:
#"Ver cargo y empresa"
# Adaptando select("item", "valor") a tus datos
df_filtrado.select("puesto", "empresa").show(5, truncate=False)

+--------------------------------------------------------------------+--------------------------------------------+
|puesto                                                              |empresa                                     |
+--------------------------------------------------------------------+--------------------------------------------+
|Developer Senior Android (SDK) & TypeScript Specialist Freelance    |FoodTag Santiago (Hybrid)                   |
|Java Developer Senior Full time                                     |Improving South America Remote (5 locations)|
|Developer Back-end Node.js Senior (Latam) Full time                 |BC Tecnología Remote                        |
|Full-Stack Engineer — Document Generation & Report Systems Full time|Blazestack Remote                           |
|Dev Integraciones Golang+Java Full time                             |Coderslab.io Remote                         |
+--------------------------------------------------------------------+--

In [12]:
#Reporte Estadístico
from pyspark.sql import functions as F

# Reporte consolidado por Nivel
reporte_final = df_filtrado.groupBy("nivel_corregido").agg(
    F.count("puesto").alias("total_ofertas"),
    F.collect_set("modalidad_simple").alias("modalidades_disponibles")
).orderBy(F.desc("total_ofertas"))

print("ANALISIS DE MERCADO LABORAL POR NIVEL:")
reporte_final.show(truncate=False)

ANALISIS DE MERCADO LABORAL POR NIVEL:
+-------------------+-------------+------------------------------+
|nivel_corregido    |total_ofertas|modalidades_disponibles       |
+-------------------+-------------+------------------------------+
|Mid/Sin especificar|93           |[Por definir, Remoto, Híbrido]|
|Senior             |77           |[Por definir, Remoto, Híbrido]|
|Lead/Architect     |6            |[Remoto, Híbrido]             |
|Junior             |2            |[Por definir, Remoto]         |
+-------------------+-------------+------------------------------+



In [13]:
from pyspark.sql import functions as F

# 1. Creamos una columna numérica temporal para poder ordenar
# Extraemos el primer número que aparezca en la columna 'salario'
df_con_sueldo = df_pro.withColumn(
    "salario_num", 
    F.regexp_extract(F.col("salario"), r"(\d+)", 1).cast("double")
)

# 2. Ordenar por salario de forma descendente y tomar el primero
# Esto cumple exactamente la lógica de la guía
ganador = df_con_sueldo.orderBy(F.desc("salario_num")).limit(1)

# 3. Mostrar los resultados solicitados por la misión
print("--- RESULTADO DE LA MISIÓN 6.5.1 ---")
ganador.select(
    F.col("puesto").alias("¿Cómo se llama el ítem?"),
    F.col("nivel_corregido").alias("¿A qué categoría pertenece?"),
    F.col("fecha_captura").alias("¿En qué fecha y hora exacta se capturó?"),
    F.col("salario").alias("Valor detectado")
).show(truncate=False)

--- RESULTADO DE LA MISIÓN 6.5.1 ---
+------------------------------------+---------------------------+---------------------------------------+---------------------+
|¿Cómo se llama el ítem?             |¿A qué categoría pertenece?|¿En qué fecha y hora exacta se capturó?|Valor detectado      |
+------------------------------------+---------------------------+---------------------------------------+---------------------+
|Senior Front-end Developer Freelance|Senior                     |2026-04-08 13:57:26                    |1400 - 2000 USD/month|
+------------------------------------+---------------------------+---------------------------------------+---------------------+



In [15]:
from pyspark.sql import functions as F

# 1. Definimos la variable aquí mismo para que no de NameError
NOMBRE_GRUPO = "mercado laboral"

# 2. Ejecutamos el Ticket de Salida
# Asegúrate de haber corrido las celdas anteriores donde creamos 'df_pro'
ticket_salida = df_pro.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("nivel_corregido") \
    .agg(
        F.count("puesto").alias("total_ofertas"),
        F.count("salario").alias("registros_con_sueldo"), 
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: mercado laboral ---
+-------------------+-------------+--------------------+---------------------+
|    nivel_corregido|total_ofertas|registros_con_sueldo|ultima_sincronizacion|
+-------------------+-------------+--------------------+---------------------+
|             Junior|            2|                   2|  2026-04-08 13:57:53|
|     Lead/Architect|            6|                   6|  2026-04-08 13:57:52|
|Mid/Sin especificar|           93|                  93|  2026-04-08 13:58:01|
|             Senior|           77|                  77|  2026-04-08 13:58:00|
+-------------------+-------------+--------------------+---------------------+

